In [ ]:
import os
os.environ['OLLAMA_HOST'] = 'http://ollama:11434'
import litellm
litellm.api_base = 'http://ollama:11434'
import requests
try:
    response = requests.get('http://ollama:11434/api/tags')
    models = response.json()['models'] 
    print(f" Connected to shared Ollama!")
    print(f" Available models: {len(models)}")
    for model in models:
        print(f"   - {model['name']}")
except Exception as e:
    print(f"Error - Ollama connection: {e}")
print("\n PDL Notebook Ready!")
%load_ext pdl.pdl_notebook_ext


<div style="text-align: center; padding: 60px 20px;">
<h1 style="font-size: 52px; margin-bottom: 15px; color: #2c3e50; font-weight: 300; letter-spacing: 1px;">PDL: Prompt Declaration Language</h1>
<p style="text-align: center; font-size: 26px; color: #7f8c8d; margin-bottom: 10px; font-weight: 300;">Reliable & Structured LLM Interactions for Code & Beyond</p>
<p style="text-align: center; font-size: 20px; color: #7f8c8d; margin-bottom: 10px; font-weight: 400;">By <strong>Tim Hayes</strong></p>
<p style="text-align: center; font-size: 18px; color: #7f8c8d; margin-bottom: 10px;">Supervisor: <strong>Prof. Manuel Maarek</strong></p>
</div>

<div style="padding: 40px; max-width: 1000px; margin: 0 auto; font-size: 18px; line-height: 1.9;">
<h2 style="text-align: center; font-size: 32px; margin-bottom: 40px; color: #2c3e50;">Learning Outcomes</h2>
<ul style="list-style: none; padding: 0; border-left: 5px solid #3498db; padding-left: 30px;">
<li style="margin-bottom: 18px; padding-left: 15px;">Understand why prompt reliability is hard in production</li>
<li style="margin-bottom: 18px; padding-left: 15px;">Learn PDL core concepts and authoring style</li>
<li style="margin-bottom: 18px; padding-left: 15px;">Compare PDL with LangChain, DSPy, LMQL, and PromptL</li>
<li style="margin-bottom: 18px; padding-left: 15px;">Know when PDL is the right tool for your system</li>
</ul>
</div>

---

<div style="padding: 40px; max-width: 1040px; margin: 0 auto;">
<h1 style="font-size: 40px; margin-bottom: 14px; color: #2c3e50;">SECTION 1: Introduction to PDL</h1>
<h2 style="font-size: 31px; color: #34495e; margin-bottom: 20px;">1.1: Why Regular String Prompting Breaks</h2>
<p style="font-size: 17px; color: #7f8c8d; line-height: 1.75; margin-bottom: 14px;">String prompting is fine for quick demos. It becomes unreliable once you move to multi-step workflows and production traffic.</p>
<div style="background-color: #f8f9fa; padding: 24px; border-radius: 8px; border-left: 5px solid #e74c3c; margin-top: 16px;">
<pre style="color: #f8f8f2; padding: 16px; border-radius: 5px; overflow-x: auto; font-size: 14px; margin: 0; border: 1px solid #555;">
<code>prompt = "Extract invoice fields as JSON"
response = llm.generate(prompt)
# Works on day 1
# Breaks later when format drifts or fields go missing</code>
</pre>
</div>
</div>

<div style="padding: 40px; max-width: 1040px; margin: 0 auto;">
<h2 style="font-size: 32px; color: #34495e; margin-bottom: 24px;">1.2: Problems with Regular Prompting</h2>
<div style="padding: 20px; background-color: #f8f9fa; border-left: 5px solid #e74c3c; border-radius: 8px; margin-bottom: 14px;">
<p style="font-size: 17px; margin: 0 0 10px 0; color: #2c3e50;">
<strong>Lack of Structure</strong></p>
<ul style="font-size: 16px; line-height: 1.85; margin: 0; padding-left: 22px; color: #2c3e50;">
<li>Prompts are unstructured strings</li>
<li>No formal specification of expected output</li>
<li>Developers end up guessing the output format</li><li>Often this becomes: pray the LLM returns valid JSON</li>
</ul>
</div>
<div style="padding: 20px; background-color: #f8f9fa; border-left: 5px solid #f39c12; border-radius: 8px; margin-bottom: 14px;">
<p style="font-size: 17px; margin: 0 0 10px 0; color: #2c3e50;">
<strong>Reusability Issues</strong></p>
<ul style="font-size: 16px; line-height: 1.85; margin: 0; padding-left: 22px; color: #2c3e50;">
<li>Prompt engineering is manual and tedious</li>
<li>Copy-pasting prompts across projects is common</li>
<li>Consistency is difficult to maintain across teams</li>
</ul>
</div>
</div>

---

<div style="padding: 40px; max-width: 1040px; margin: 0 auto;"><h2 style="font-size: 32px; margin-bottom: 22px; color: #2c3e50;">1.3: The PDL Solution</h2><div style="padding: 24px; background-color: #eafaf1; border-left: 5px solid #27ae60; border-radius: 8px; margin-bottom: 14px;"><p style="font-size: 16px; line-height: 1.8; margin: 0; color: #2c3e50;"><strong>PDL is a declarative, YAML-based language that formally specifies and reliably composes LLM interactions.</strong></p></div><div style="padding: 20px; background-color: #f8f9fa; border-left: 5px solid #27ae60; border-radius: 8px;"><p style="font-size: 17px; margin: 0 0 10px 0; color: #2c3e50;"><strong>Key Benefits</strong></p><ul style="font-size: 16px; line-height: 1.9; margin: 0; padding-left: 22px; color: #2c3e50;"><li><strong>YAML-based:</strong> readable, familiar for most engineers, and easy to version-control</li><li><strong>Formal specification:</strong> clearly defines what goes in and what must come out</li><li><strong>Composable calls:</strong> chain steps like generate, review, and fix in one workflow</li><li><strong>Type checking + validation:</strong> parser/spec ensures valid structure or fails immediately</li></ul></div></div>

<div style="padding: 40px; max-width: 1040px; margin: 0 auto;"><h2 style="font-size: 32px; margin-bottom: 22px; color: #2c3e50;">1.4: Design Targets for PDL</h2><div style="padding: 24px; background-color: #fef5e7; border-left: 5px solid #f39c12; border-radius: 6px;"><ul style="font-size: 16px; line-height: 1.95; margin: 0; padding-left: 22px; color: #2c3e50;"><li>Make prompt logic readable and reviewable</li><li>Add strict structured output checks (parser/spec)</li><li>Support loops, branching, variables, and reuse in one place</li><li>Cut down hidden behavior spread across app code</li></ul></div></div>

---

<div style="padding: 40px; max-width: 1040px; margin: 0 auto;"><h2 style="font-size: 32px; margin-bottom: 20px; color: #2c3e50;">1.5: String Prompting vs PDL</h2><p style="font-size: 17px; color: #7f8c8d; line-height: 1.7; margin-bottom: 16px;">Same goal. Very different reliability once systems scale.</p><div style="overflow-x: auto;"><table style="width: 100%; border-collapse: collapse; font-size: 15px;"><tr style="background-color: #2c3e50; color: white;"><td style="padding: 12px; font-weight: bold;">Area</td><td style="padding: 12px; font-weight: bold;">String Prompting</td><td style="padding: 12px; font-weight: bold;">PDL</td></tr><tr style="border-bottom: 1px solid #ddd;"><td style="padding: 12px; font-weight: bold;">Logic visibility</td><td style="padding: 12px;">Mostly hidden</td><td style="padding: 12px;">Explicit blocks</td></tr><tr style="border-bottom: 1px solid #ddd;"><td style="padding: 12px; font-weight: bold;">Output safety</td><td style="padding: 12px;">Manual checks</td><td style="padding: 12px;">parser/spec checks</td></tr><tr style="border-bottom: 1px solid #ddd;"><td style="padding: 12px; font-weight: bold;">Reuse</td><td style="padding: 12px;">Copy/paste</td><td style="padding: 12px;">Functions/defs</td></tr><tr><td style="padding: 12px; font-weight: bold;">Maintenance</td><td style="padding: 12px;">High overhead</td><td style="padding: 12px;">Lower overhead</td></tr></table></div></div>

<div style="text-align: center; padding: 50px 20px;"><h1 style="font-size: 44px; margin-bottom: 10px; color: #2c3e50; font-weight: 400;">SECTION 2: Comparing Alternatives</h1><p style="text-align: center; font-size: 18px; color: #7f8c8d; margin: 0;">Five frameworks: where they work and where they don't</p><div style="width: 120px; height: 3px; background-color: #3498db; margin: 25px auto;"></div></div>

---

<div style="padding: 40px; max-width: 1100px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 20px; color: #2c3e50;">Quick Look: What Each Framework Does</h2>
<div style="overflow-x: auto;">
<table style="width: 100%; border-collapse: collapse; font-size: 15px;">
<tr style="background-color: #2c3e50; color: white;">
<td style="padding: 12px; font-weight: bold;">Framework</td>
<td style="padding: 12px; font-weight: bold;">Good For</td>
<td style="padding: 12px; font-weight: bold;">How It Works</td>
<td style="padding: 12px; font-weight: bold;">Output Checking</td>
<td style="padding: 12px; font-weight: bold;">Multi-Step Support</td>
</tr>
<tr style="border-bottom: 1px solid #ddd;">
<td style="padding: 12px; font-weight: bold; color: #f39c12;">LangChain</td>
<td style="padding: 12px;">Quick prototyping</td>
<td style="padding: 12px;">Python code</td>
<td style="padding: 12px;">Manual</td>
<td style="padding: 12px;">Yes, via code</td>
</tr>
<tr style="border-bottom: 1px solid #ddd;">
<td style="padding: 12px; font-weight: bold; color: #e74c3c;">DSPy</td>
<td style="padding: 12px;">Tuning & optimization</td>
<td style="padding: 12px;">Python modules</td>
<td style="padding: 12px;">Minimal</td>
<td style="padding: 12px;">Not really</td>
</tr>
<tr style="border-bottom: 1px solid #ddd;">
<td style="padding: 12px; font-weight: bold; color: #8e44ad;">LMQL</td>
<td style="padding: 12px;">Locked-in outputs</td>
<td style="padding: 12px;">Query language</td>
<td style="padding: 12px;">Built-in</td>
<td style="padding: 12px;">Not really</td>
</tr>
<tr style="border-bottom: 1px solid #ddd;">
<td style="padding: 12px; font-weight: bold; color: #f1c40f;">PromptL</td>
<td style="padding: 12px;">Basic templates</td>
<td style="padding: 12px;">Template engine</td>
<td style="padding: 12px;">None</td>
<td style="padding: 12px;">No</td>
</tr>
<tr>
<td style="padding: 12px; font-weight: bold; color: #27ae60;">PDL</td>
<td style="padding: 12px;">Production systems</td>
<td style="padding: 12px;">YAML declarative</td>
<td style="padding: 12px;">Built-in</td>
<td style="padding: 12px;">Yes, fully</td>
</tr>
</table>
</div>
</div>

---

<div style="padding: 40px; max-width: 1040px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 20px; color: #2c3e50;">2.1: LangChain</h2>
<div style="padding: 24px; background-color: #fef5e7; border-left: 5px solid #f39c12; border-radius: 8px;">
<p style="font-size: 15px; color: #2c3e50; line-height: 1.8; margin: 0 0 10px 0;"><strong>What it is:</strong> Python framework that lets you build apps fast with lots of integrations.</p>
<pre style="background-color: #f8f9fa; color: #2c3e50; padding: 14px; border-radius: 6px; font-size: 13px; overflow-x: auto; margin: 0 0 10px 0; border: 1px solid #555;"><code><span style="color: #7f8c8d;">// You write the flow directly in code</span>
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("Summarize: {text}")
chain = prompt | ChatOpenAI(model="gpt-4o-mini")
result = chain.invoke({"text": report_text})</code></pre>
<p style="font-size: 15px; margin: 0 0 8px 0; color: #2c3e50; line-height: 1.8;"><strong>Why use it:</strong></p>
<ul style="font-size: 14px; line-height: 1.8; margin: 0 0 10px 0; padding-left: 22px; color: #2c3e50;">
<li>Build things quickly in Python</li>
<li>Lots of integrations ready to go</li>
<li>Flexible and easy to modify</li>
</ul>
<p style="font-size: 15px; margin: 0; color: #e74c3c; line-height: 1.8;"><strong>Problem:</strong> No built-in way to validate outputs. You have to add that yourself.</p>
<p style="font-size: 15px; margin: 10px 0 0 0; color: #2c3e50; line-height: 1.8;"><strong>Compared to PDL:</strong> LangChain gives faster app scaffolding, but PDL gives stronger built-in workflow contracts and output validation.</p>
</div>
</div>

---

<div style="padding: 40px; max-width: 1100px; margin: 0 auto;">
<h3 style="font-size: 24px; margin-bottom: 16px; color: #34495e;">2.2: DSPy</h3>
<div style="padding: 22px; background-color: #fdeaea; border-left: 4px solid #e74c3c; border-radius: 6px;">
<p style="font-size: 15px; margin: 0 0 10px 0; color: #2c3e50; line-height: 1.8;"><strong>What it is:</strong> Python framework that helps you tune and improve your prompts automatically using benchmarks.</p>
<pre style="background-color: #f8f9fa; color: #2c3e50; padding: 14px; border-radius: 6px; font-size: 13px; overflow-x: auto; margin: 0 0 10px 0; border: 1px solid #555;"><code>import dspy

class Correct(dspy.Signature):
    text = dspy.InputField()
    corrected = dspy.OutputField()

program = dspy.Predict(Correct)
result = program(text="she go to school yesterday")</code></pre>
<p style="font-size: 15px; margin: 0 0 8px 0; color: #2c3e50; line-height: 1.8;"><strong>Why use it:</strong></p>
<ul style="font-size: 14px; line-height: 1.8; margin: 0 0 10px 0; padding-left: 22px; color: #2c3e50;">
<li>You have test data and metrics</li>
<li>You want to automatically improve quality</li>
<li>Systematic tuning loops</li>
</ul>
<p style="font-size: 15px; margin: 0; color: #e74c3c; line-height: 1.8;"><strong>Problem:</strong> Only focuses on making things better. Doesn't help with building complex workflows.</p>
<p style="font-size: 15px; margin: 10px 0 0 0; color: #2c3e50; line-height: 1.8;"><strong>Compared to PDL:</strong> DSPy is stronger for optimization loops; PDL is stronger for declaring and validating end-to-end multi-step runtime behavior.</p>
</div>
</div>

---

<div style="padding: 40px; max-width: 1100px; margin: 0 auto;">
<h3 style="font-size: 24px; margin-bottom: 16px; color: #34495e;">2.3: LMQL</h3>
<div style="padding: 22px; background-color: #f4ecf7; border-left: 4px solid #8e44ad; border-radius: 6px;">
<p style="font-size: 15px; margin: 0 0 10px 0; color: #2c3e50; line-height: 1.8;"><strong>What it is:</strong> A language that lets you control exactly what the model outputs, down to the token level.</p>
<pre style="background-color: #f8f9fa; color: #2c3e50; padding: 14px; border-radius: 6px; font-size: 13px; overflow-x: auto; margin: 0 0 10px 0; border: 1px solid #555;"><code>argmax
  "Classify sentiment. Label: [label]"
where
  label in ["positive", "neutral", "negative"]</code></pre>
<p style="font-size: 15px; margin: 0 0 8px 0; color: #2c3e50; line-height: 1.8;"><strong>Why use it:</strong></p>
<ul style="font-size: 14px; line-height: 1.8; margin: 0 0 10px 0; padding-left: 22px; color: #2c3e50;">
<li>You need outputs that are always valid</li>
<li>You want tight control over what comes out</li>
<li>Great for classification and structured responses</li>
</ul>
<p style="font-size: 15px; margin: 0; color: #e74c3c; line-height: 1.8;"><strong>Problem:</strong> Can only handle simple cases. Not good for multi-step workflows.</p>
<p style="font-size: 15px; margin: 10px 0 0 0; color: #2c3e50; line-height: 1.8;"><strong>Compared to PDL:</strong> LMQL is stronger for tight token constraints, while PDL is stronger when that constrained step is part of a larger workflow.</p>
</div>
</div>

---

<div style="padding: 40px; max-width: 1100px; margin: 0 auto;">
<h3 style="font-size: 24px; margin-bottom: 16px; color: #34495e;">2.4: PromptL</h3>
<div style="padding: 22px; background-color: #fef5e7; border-left: 4px solid #f39c12; border-radius: 6px;">
<p style="font-size: 15px; margin: 0 0 10px 0; color: #2c3e50; line-height: 1.8;"><strong>What it is:</strong> A simple tool for storing and reusing prompt templates.</p>
<pre style="background-color: #f8f9fa; color: #2c3e50; padding: 14px; border-radius: 6px; font-size: 13px; overflow-x: auto; margin: 0 0 10px 0; border: 1px solid #555;"><code>template "ad_copy":
  "Write a short ad for {product} in a {tone} tone."

render("ad_copy", product="running shoes", tone="friendly")</code></pre>
<p style="font-size: 15px; margin: 0 0 8px 0; color: #2c3e50; line-height: 1.8;"><strong>Why use it:</strong></p>
<ul style="font-size: 14px; line-height: 1.8; margin: 0 0 10px 0; padding-left: 22px; color: #2c3e50;">
<li>Very easy to learn</li>
<li>Good if you have lots of similar prompts</li>
<li>Simple variable swapping</li>
</ul>
<p style="font-size: 15px; margin: 0; color: #e74c3c; line-height: 1.8;"><strong>Problem:</strong> That's basically all it does. No control flow or validation at all.</p>
<p style="font-size: 15px; margin: 10px 0 0 0; color: #2c3e50; line-height: 1.8;"><strong>Compared to PDL:</strong> PromptL is lighter for template reuse, but PDL adds control flow, state, and typed validation for production workflows.</p>
</div>
</div>

---

<div style="padding: 40px; max-width: 1100px; margin: 0 auto;">
<h3 style="font-size: 24px; margin-bottom: 16px; color: #34495e;">2.5: PDL</h3>
<div style="padding: 22px; background-color: #eafaf1; border-left: 5px solid #27ae60; border-radius: 8px;">
<p style="font-size: 15px; color: #2c3e50; line-height: 1.8; margin: 0 0 10px 0;"><strong>What it is:</strong> A language where you write your entire workflow in YAML. Everything is clear and explicit.</p>
<pre style="background-color: #f8f9fa; color: #2c3e50; padding: 14px; border-radius: 6px; font-size: 13px; overflow-x: auto; margin: 0 0 10px 0; border: 1px solid #555;"><code>defs:
  min_income: 32000
if: ${applicant.income >= min_income}
then:
  model: ollama/granite-code:3b
  parser: json
  spec:
    approved: boolean
    reason: string</code></pre>
<p style="font-size: 15px; margin: 0 0 8px 0; color: #2c3e50; line-height: 1.8;"><strong>Why use it:</strong></p>
<ul style="font-size: 14px; line-height: 1.8; margin: 0 0 10px 0; padding-left: 22px; color: #2c3e50;">
<li>Your workflow is written down, not hidden in prompts</li>
<li>Validates outputs automatically</li>
<li>Reusable functions and variables</li>
<li>Everything in one place</li>
</ul>
</div>
</div>

<div style="text-align: center; padding: 60px 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 36px; margin-bottom: 30px; color: #2c3e50;">2.6 Summary of Use-Cases</h2>
<div style="text-align: left; padding: 24px; background-color: #ecf0f1; border-radius: 8px;">
<p style="font-size: 16px; color: #2c3e50; line-height: 1.9; margin-bottom: 15px;"><strong style="color: #27ae60;">Use PDL</strong> if you're building something serious that needs to work reliably. Best for production stuff with multiple steps and strict output checking.</p>
<p style="font-size: 16px; color: #2c3e50; line-height: 1.9; margin-bottom: 15px;"><strong style="color: #f39c12;">Use LangChain</strong> if you just want to build something fast. Best for trying things out and experimenting.</p>
<p style="font-size: 16px; color: #2c3e50; line-height: 1.9; margin-bottom: 15px;"><strong style="color: #e74c3c;">Use DSPy</strong> if you have test data and want to automatically make things better.</p>
<p style="font-size: 16px; color: #2c3e50; line-height: 1.9; margin-bottom: 15px;"><strong style="color: #8e44ad;">Use LMQL</strong> if you need outputs that are always exactly right.</p>
<p style="font-size: 16px; color: #2c3e50; line-height: 1.9;"><strong style="color: #f1c40f;">Use PromptL</strong> if you're just storing prompts and swapping variables.</p>
</div>
</div>

<div style="text-align: center; padding: 50px 20px;">
<h1 style="font-size: 44px; margin-bottom: 10px; color: #2c3e50; font-weight: 400;">SECTION 3: PDL Fundamentals</h1>
<p style="text-align: center; font-size: 18px; color: #7f8c8d; margin: 0;">Core Concepts and Practical Demonstrations</p>
<div style="width: 120px; height: 3px; background-color: #3498db; margin: 25px auto;"></div>
</div>

---

<div style="padding: 40px; max-width: 1100px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 15px; color: #2c3e50;">3.1: Three-Layer Architecture</h2>
<p style="font-size: 17px; color: #7f8c8d; margin-bottom: 30px;">This is the core concept of how PDL works.</p>

<div style="background-color: #f0f0f0; padding: 30px; border-radius: 8px; font-family: 'Courier New', monospace; font-size: 13px; line-height: 1.6; overflow-x: auto; margin-bottom: 35px; border: 1px solid #ddd;">
<pre style="margin: 0; text-align: center">┌─────────────────────────────────────────────────────────────────────┐
│ LAYER 3: LLM Call                                                   │
│ Uses accumulated context as prompt                                  │
└──────────────────────────────┬──────────────────────────────────────┘
                               ↑ Passes context
┌──────────────────────────────┴──────────────────────────────────────┐
│ LAYER 2: Background Context                                         │
│ Accumulates all previous outputs                                    │
│ [system, user, assistant, user, assistant...]                       │
└──────────────────────────────┬──────────────────────────────────────┘
                               ↑ Data flows up
┌──────────────────────────────┴──────────────────────────────────────┐
│ LAYER 1: Individual Blocks                                          │
│ text:, model:, if:, for:, defs: ...                                 │
│ Each block produces data or logic                                   │
└─────────────────────────────────────────────────────────────────────┘</pre>
</div>

<div style="display: grid; grid-template-columns: 2fr 1fr; gap: 30px;">
<div>
<h3 style="font-size: 22px; margin-bottom: 20px; color: #34495e;\">Execution Flow</h3>
<ol style="font-size: 18px; line-height: 2.2; margin: 0; padding-left: 25px;">
<li>Execute <code>text:</code> block → added to context</li>
<li>Execute <code>text:</code> block → added to context</li>
<li>Execute <code>model:</code> block → LLM sees <strong>everything above</strong></li>
<li>LLM response → added to context</li>
<li>Next block continues...</li>
</ol>
</div>
</div>
</div>

---

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;"><h2 style="font-size: 32px; margin-bottom: 15px; color: #2c3e50;">3.2: Understanding Context Accumulation</h2><div style="padding: 25px; background-color: #e8f4f8; border-radius: 8px; border-left: 5px solid #2980b9; margin-bottom: 25px;"><h3 style="font-size: 20px; margin-top: 0; margin-bottom: 12px; color: #165fa0;">What is Context Accumulation?</h3><p style="font-size: 16px; color: #1e5f9e; margin: 0; line-height: 1.7;"><strong>Context accumulation</strong> is the process where PDL automatically combines all previous text blocks and LLM responses into a single conversation history that gets sent to the next LLM call. Instead of you manually managing what to include in each prompt, PDL remembers everything and makes it available automatically.</p></div></div>

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 15px; color: #2c3e50;">3.2: With and Without PDL</h2>
<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 25px; margin-bottom: 25px;">
<div style="padding: 20px; background-color: #fdeaea; border-radius: 8px; border-left: 4px solid #e74c3c;">
<h4 style="font-size: 16px; margin-top: 0; margin-bottom: 10px; color: #c0392b;">Without PDL (Manual)</h4>
<pre style="margin: 0; font-size: 12px; color: #2c3e50; background-color: #fff;">
<code>context = "You are helpful\n"
    response1 = llm(context)
    context += response1
    context += "Next question?"
    response2 = llm(context)  
    # Manual!
</code>
</pre>
</div>
<div style="padding: 20px; background-color: #eafaf1; border-radius: 8px; border-left: 4px solid #27ae60;">
<h4 style="font-size: 16px; margin-top: 0; margin-bottom: 10px; color: #196f3d;">With PDL (Automatic)</h4>
<pre style="margin: 0; font-size: 12px; color: #2c3e50; background-color: #fff;">
<code>text:- "You are helpful\n"- 
model: llm- "Next question?\n"- 
model: llm  
# Auto uses all!</code>
</pre>
</div>
</div>
<p style="font-size: 16px; color: #7f8c8d; margin-bottom: 0; line-height: 1.7;">
<strong>Use Case:</strong> 
This demonstrates how PDL automatically manages conversation state. You write blocks in order, and PDL ensures each LLM call sees everything that came before it. This eliminates a huge source of bugs and cognitive overhead.</p>
</div>

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;"><h2 style="font-size: 32px; margin-bottom: 15px; color: #2c3e50;">3.2: Code Demonstration - Context Accumulation</h2><p style="font-size: 17px; color: #7f8c8d; margin-bottom: 0; line-height: 1.6;">Run this PDL program to see context accumulation in practice across sequential text and model blocks.</p></div>

In [ ]:
%%pdl --reset-context
description: Context accumulation demo
text:
- "You are an expert Python programmer."
- "You help users write clean, efficient code."
- "User: Write a function to check if a number is prime. Please include comments.\n"
- model: ollama/granite-code:3b
  parameters:
    temperature: 0.7
    max_tokens: 200


---

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 15px; color: #2c3e50;">3.3: Demo - Conditional Logic</h2>
<p style="font-size: 17px; color: #7f8c8d; margin-bottom: 25px; line-height: 1.6;"><strong>Use Case:</strong> Control program flow based on conditions. Execute different PDL blocks depending on variable values, comparison results, or dynamic data.</p>
</div>

In [ ]:
%%pdl --reset-context
description: Conditional logic with scoring

if: ${85 > 80}
then:
  text: "Excellent score! (85 > 80)"
else:
  text: "Needs improvement (85 <= 80)"


---

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 15px; color: #2c3e50;">3.4: Control Flow - Loops</h2>
<p style="font-size: 17px; color: #7f8c8d; margin-bottom: 25px; line-height: 1.6;"><strong>Use Case:</strong> Repeat blocks of code multiple times. PDL supports both counted loops (repeat N times) and list iteration (for each item). Loops are useful for batch processing, generating multiple variations, or iterating over datasets.</p>
</div>

In [ ]:
%%pdl --reset-context
description: Counted repeat loop with join

text:
- "Process items:\n"
- repeat: "- Item ${i}\n"
  index: i
  maxIterations: 3
  join:
    with: ""

---

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 15px; color: #2c3e50;">3.4.5: For Loops - Iterating Over Lists</h2>
<p style="font-size: 17px; color: #7f8c8d; margin-bottom: 25px; line-height: 1.6;"><strong>Use Case:</strong> Known collection iteration. When you have a fixed list of items, use `for` loops to iterate over each element and execute the same block.</p>
</div>

In [ ]:
%%pdl --reset-context
description: For loop over a list

defs:
  languages:
    data: ["Python", "JavaScript", "Go"]

for:
  lang: ${languages}
repeat:
  text: "- ${lang}\n"


---

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 18px; color: #2c3e50;">3.5: Template Variables with <code>def:</code> and <code>defs:</code></h2>

<h3 style="font-size: 22px; margin-bottom: 10px; color: #34495e;">Concept</h3>
<p style="font-size: 16px; color: #7f8c8d; margin: 0; line-height: 1.7;"> Template variables (<code>${...}</code>) let you carry information forward instead of rewriting prompts over and over. Use <code>def:</code> when the value is generated at runtime by a block, and use <code>defs:</code> when the value is known upfront and should be reused across the workflow.</p>
</div>

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 20px; color: #2c3e50;">3.5: Template Variables with <code>def:</code> and <code>defs:</code></h2>

<h3 style="font-size: 22px; margin-bottom: 10px; color: #34495e;"><code>def:</code> - Store Runtime Output</h3>
<pre style="background-color: #f8f9fa; color: #2c3e50; padding: 16px; border-radius: 6px; font-size: 14px; overflow-x: auto; margin-bottom: 12px; border: 1px solid #555;"><code>- model: ollama/granite-code:3b
  def: draft
- "Refine this: ${draft}"</code></pre>
<p style="font-size: 16px; color: #7f8c8d; margin: 0; line-height: 1.7;"><strong>What it does:</strong> <code>def:</code> captures the output of a block and binds it to a variable for later reuse. It is best when a generated value should directly influence a later prompt step.</p>
</div>

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 20px; color: #2c3e50;">3.5: Template Variables with <code>def:</code> and <code>defs:</code></h2>

<h3 style="font-size: 22px; margin-bottom: 10px; color: #34495e;"><code>defs:</code> - Predefine Reusable Values</h3>
<pre style="background-color: #f8f9fa; color: #2c3e50; padding: 16px; border-radius: 6px; font-size: 14px; overflow-x: auto; margin-bottom: 12px; border: 1px solid #555;"><code>defs:
  task: "sorting"
  tone: "concise"

text:
\- "Task: \${task}, tone: \${tone}\n"</code></pre>
<p style="font-size: 16px; color: #7f8c8d; margin: 0; line-height: 1.7;"><strong>What it does:</strong> <code>defs:</code> declares static or reusable values up front. Use it for shared constants, labels, and configuration knobs that multiple blocks should reference.</p>
</div>

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 20px; color: #2c3e50;">3.5: Template Variables with <code>def:</code> and <code>defs:</code></h2>

<h3 style="font-size: 22px; margin-bottom: 12px; color: #34495e;">Quick Difference</h3>
<div style="padding: 20px; background-color: #f8f9fa; border-radius: 8px; border-left: 5px solid #3498db; overflow-x: auto;">
<table style="width: 100%; border-collapse: collapse; font-size: 15px;">
<tr style="background-color: #e8f4f8; border-bottom: 2px solid #3498db;">
<td style="padding: 10px; font-weight: bold; color: #2980b9;"></td>
<td style="padding: 10px; font-weight: bold; color: #2980b9;">def:</td>
<td style="padding: 10px; font-weight: bold; color: #2980b9;">defs:</td>
</tr>
<tr style="border-bottom: 1px solid #ddd;">
<td style="padding: 10px; font-weight: bold; color: #2c3e50;">Value source</td>
<td style="padding: 10px;">Produced by the current block</td>
<td style="padding: 10px;">Declared manually in setup</td>
</tr>
<tr style="border-bottom: 1px solid #ddd;">
<td style="padding: 10px; font-weight: bold; color: #2c3e50;">Typical role</td>
<td style="padding: 10px;">Carry model output forward</td>
<td style="padding: 10px;">Provide reusable constants</td>
</tr>
<tr>
<td style="padding: 10px; font-weight: bold; color: #2c3e50;">When to choose</td>
<td style="padding: 10px;">Data is generated at runtime</td>
<td style="padding: 10px;">Data is known before execution</td>
</tr>
</table>
</div>
</div>

In [ ]:
%%pdl --reset-context
description: Variable definition with def and defs blocks

defs:
  system_prompt: "You are a helpful Python expert."
  code_style: "clean and Pythonic"

text:
- "${system_prompt}\n"
- "Write a simple function with ${code_style} style.\n"
- model: ollama/granite-code:3b
  parameters:
    temperature: 0.7
    max_tokens: 100
  def: generated_code
- "\n"
- "Generated code stored in variable:\n"
- "${generated_code}\n"
- "\n"
- "Now test it:\n"
- model: ollama/granite-code:3b
  parameters:
    temperature: 0.3
    max_tokens: 100
  def: test_result
- "Test result:\n${test_result}\n"

---

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 18px; color: #2c3e50;">3.6: Structured Output with Parser & Spec</h2>

<h3 style="font-size: 22px; margin-bottom: 10px; color: #34495e;">Concept</h3>
<p style="font-size: 16px; color: #7f8c8d; margin: 0; line-height: 1.7;">Normally with regular prompting, you ask for JSON and just hope the model behaves. With <code>parser: json</code>, PDL extracts the structured payload, and with <code>spec:</code>, it validates exactly what fields and types must exist.</p>
</div>

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 20px; color: #2c3e50;">3.6: Structured Output with Parser & Spec</h2>

<h3 style="font-size: 22px; margin-bottom: 14px; color: #34495e;">Before vs After: Without and With Validation</h3>

<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 18px;">
<div style="padding: 16px; background-color: #fdeaea; border-radius: 8px; border-left: 4px solid #e74c3c;">
<p style="font-size: 14px; font-weight: bold; margin: 0 0 10px 0; color: #c0392b;">WITHOUT Parser (Unreliable)</p>
<pre style="background-color: #f8f9fa; color: #2c3e50; padding: 14px; border-radius: 6px; font-size: 12px; overflow-x: auto; margin: 0; border: 1px solid #555;"><code>- model: ollama/granite-code:3b
  text: "Extract user info from: John, john@example.com. Return JSON: {name, email}"

May return extra prose or wrong shape
Manual parsing and checks required</code></pre>
</div>

<div style="padding: 16px; background-color: #eafaf1; border-radius: 8px; border-left: 4px solid #27ae60;">
<p style="font-size: 14px; font-weight: bold; margin: 0 0 10px 0; color: #196f3d;">WITH Parser + Spec (Reliable)</p>
<pre style="background-color: #f8f9fa; color: #2c3e50; padding: 14px; border-radius: 6px; font-size: 12px; overflow-x: auto; margin: 0; border: 1px solid #555;"><code>- model: ollama/granite-code:3b
  parser: json
  spec:
    name: string
    email: string
  def: user_data

Returns typed object or fails clearly</code></pre>
</div>
</div>

<p style="font-size: 16px; color: #7f8c8d; margin: 14px 0 0 0; line-height: 1.7;"><strong>Takeaway:</strong> parser/spec shifts JSON handling from brittle post-processing to explicit validation at generation time.</p>
</div>

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 20px; color: #2c3e50;">3.6: Structured Output with Parser & Spec</h2>

<h3 style="font-size: 22px; margin-bottom: 10px; color: #34495e;">Use When</h3>
<ul style="font-size: 16px; line-height: 1.9; margin: 0; color: #2c3e50; padding-left: 24px;">
<li>You need structured JSON for APIs, evaluators, or typed application logic</li>
<li>You want schema validation near generation, not fragile parsing later</li>
<li>You are moving from prototype prompts to production-grade reliability</li>
</ul>
<p style="font-size: 16px; color: #7f8c8d; margin: 12px 0 0 0;"><strong>Benefits:</strong> parser/spec failures surface immediately, so downstream code receives validated data or a clear error path.</p>
</div>

In [ ]:
%%pdl --reset-context
description: Structured JSON output with validation
text:
- "Write a Python function to sort a list.\n"
- "Return ONLY valid JSON with: {\"code\": \"...\", \"explanation\": \"...\"}\n"
- model: ollama/granite-code:3b
  parameters:
    temperature: 0.1
  parser: json
  spec:
    code: string
    explanation: string
  def: result
  contribute: []
- "Generated Code:\n```python\n${result.code}\n```\n"


---

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 18px; color: #2c3e50;">3.7: Functions - Define & Reuse</h2>

<h3 style="font-size: 22px; margin-bottom: 10px; color: #34495e;">Concept</h3>
<p style="font-size: 16px; color: #7f8c8d; margin: 0; line-height: 1.7;">PDL functions let you define logic once with typed inputs, then call it anywhere with different arguments. So instead of having duplicate prompt blocks that may have slight inconcistencies, there is instead one version that is easier to maintain, test, and improve.</p>
</div>

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 20px; color: #2c3e50;">3.7: Functions - Define & Reuse</h2>

<h3 style="font-size: 22px; margin-bottom: 10px; color: #34495e;">YAML Syntax Example</h3>
<pre style="background-color: #f8f9fa; color: #2c3e50; padding: 16px; border-radius: 6px; font-size: 14px; overflow-x: auto; margin-bottom: 12px; border: 1px solid #555;"><code>defs:
  translate:
    function:
      sentence: string
      language: string
    return:
      text: "Translate ${sentence} to ${language}"

text:
\- call: ${ translate }
  args:
    sentence: "Hello"
    language: "French"</code></pre>
<p style="font-size: 16px; color: #7f8c8d; margin: 0; line-height: 1.7;"><strong>How to read it:</strong> define a function once with typed parameters, then call it with <code>args</code> anywhere in the workflow. This keeps prompt logic centralized and easier to evolve.</p>
</div>

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 20px; color: #2c3e50;">3.7: Functions - Define & Reuse</h2>

<h3 style="font-size: 22px; margin-bottom: 10px; color: #34495e;">Use When</h3>
<ul style="font-size: 16px; line-height: 1.9; margin: 0; color: #2c3e50; padding-left: 24px;">
<li>You repeat the same prompt pattern in multiple places</li>
<li>You want one canonical implementation to maintain and improve</li>
<li>You need cleaner notebooks with composable prompt building blocks</li>
</ul>
<p style="font-size: 16px; color: #7f8c8d; margin: 12px 0 0 0;"><strong>Next:</strong> Demo cell defines one review function and calls it twice with different code snippets.</p>
</div>

In [ ]:
%%pdl --reset-context
description: Reusable functions - define once, call multiple times

defs:
  review_code:
    function:
      code_snippet: string
    return:
      model: ollama/granite-code:3b
      parameters:
        temperature: 0
        max_tokens: 140
      parser: json
      spec:
        quality: number
        feedback: string
      input: |
        You are a strict Python reviewer.
        Score code from 1 to 10 using this rubric:
        - Correctness 50 percent
        - Efficiency 30 percent
        - Readability 20 percent

        Hard anchors:
        - Missing base case in recursion must score 1 to 3 overall.
        - Exponential recursion without memoization should not exceed 4 overall.
        - Using built in sorted cleanly should usually score 8 to 10 overall.
        - Do not assign the same score to clearly different snippets unless truly equivalent.

        Return ONLY valid JSON with exactly:
        {"quality": <integer 1-10>, "feedback": "<one short reason>"}

        Code snippet:
        ${code_snippet}

text:
- "=== CODE REVIEW #1: Inefficient Algorithm ===\n"
- call: ${ review_code }
  args:
    code_snippet: "def fib(n): return fib(n-1) + fib(n-2)"
  def: review_1
  contribute: []
- "Quality Score: ${review_1.quality}/10\n"
- "Feedback: ${review_1.feedback}\n\n"
- "=== CODE REVIEW #2: Efficient Sort ===\n"
- call: ${ review_code }
  args:
    code_snippet: "def sort_list(arr): return sorted(arr)"
  def: review_2
  contribute: []
- "Quality Score: ${review_2.quality}/10\n"
- "Feedback: ${review_2.feedback}\n"


---

<div style="text-align: center; padding: 50px 20px;">
<h1 style="font-size: 44px; margin-bottom: 10px; color: #2c3e50; font-weight: 400;">SECTION 4: Advanced PDL Patterns</h1>
<p style="text-align: center; font-size: 18px; color: #7f8c8d; margin: 0;">Roles, output control, input handling, and code integration</p>
<div style="width: 120px; height: 3px; background-color: #3498db; margin: 25px auto;"></div>
</div>

---

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 18px; color: #2c3e50;">4.1: Roles and Conversation Control</h2>

<h3 style="font-size: 22px; margin-bottom: 10px; color: #34495e;">Concept</h3>
<p style="font-size: 16px; color: #7f8c8d; margin: 0; line-height: 1.7;">Roles are one of the most practical controls in PDL. Instead of sending plain text and hoping for consistent behavior, you explicitly label intent with <code>system</code>, <code>user</code>, and <code>assistant</code>. The <code>system</code> role sets stable behavior, <code>user</code> carries the request, and <code>assistant</code> can preserve prior responses. This gives you predictable tone, clearer boundaries, and better multi-turn reliability.</p>
</div>

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 20px; color: #2c3e50;">4.1: Roles and Conversation Control</h2>

<h3 style="font-size: 22px; margin-bottom: 10px; color: #34495e;">YAML Syntax Example</h3>
<pre style="background-color: #f8f9fa; color: #2c3e50; padding: 16px; border-radius: 6px; font-size: 14px; overflow-x: auto; margin-bottom: 12px; border: 1px solid #555;"><code>text:
- role: system
  text: "You are a strict but helpful code reviewer."
- role: user
  text: "Review this function for readability."
- model: ollama/granite-code:3b
  parameters:
    temperature: 0.2
    max_tokens: 120</code></pre>
<p style="font-size: 16px; color: #7f8c8d; margin: 0; line-height: 1.7;"><strong>What happens:</strong> the model receives role-structured context, so behavior is guided by explicit intent rather than fragile prompt wording.</p>
</div>

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 20px; color: #2c3e50;">4.1: Roles and Conversation Control</h2>

<h3 style="font-size: 22px; margin-bottom: 12px; color: #34495e;">Use When</h3>
<ul style="font-size: 16px; line-height: 1.9; margin: 0; color: #2c3e50; padding-left: 24px;">
<li>You need consistent assistant behavior across multiple turns</li>
<li>You want clear separation between instruction, request, and response history</li>
<li>You are building workflows where tone and policy must stay stable</li>
<li>You need cleaner debugging because each message has an explicit role</li>
</ul>
<p style="font-size: 16px; color: #7f8c8d; margin: 12px 0 0 0;"><strong>Next:</strong> Demo cell shows a role-driven conversation where system guidance stays consistent while user goals change.</p>
</div>

---

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 15px; color: #2c3e50;">Demo: Role-Driven Prompting in Action</h2>
<p style="font-size: 17px; color: #7f8c8d; margin-bottom: 25px; line-height: 1.6;"><strong>Use Case:</strong> Keep a stable system instruction and run two user turns so you can see how role structure improves consistency and response control.</p>
</div>

In [ ]:
%%pdl --reset-context
description: Role-focused multi-turn generation
text:
- role: system
  text: "You are a senior software engineer. Be precise, practical, and concise."
- role: user
  text: "Explain in two sentences why parser and spec improve reliability."
- model: ollama/granite-code:3b
  parameters:
    temperature: 0.2
    max_tokens: 120
  def: first_answer
- role: user
  text: "Now rewrite that explanation for a beginner in one sentence."
- model: ollama/granite-code:3b
  parameters:
    temperature: 0.2
    max_tokens: 80
- "First answer: ${first_answer}\n"

---

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 30px; color: #2c3e50;">4.2: Output Control with <code>contribute:</code></h2>
<p style="font-size: 17px; color: #7f8c8d; margin-bottom: 22px; line-height: 1.7;"><strong>The Problem:</strong> When calling an LLM, do you want output hidden, context-only, or user-visible? By default, everything propagates everywhere. <code>contribute:</code> gives precise visibility control.</p>
<div style="overflow-x: auto;">
<table style="width: 100%; border-collapse: collapse; font-size: 15px;">
<tr style="background-color: #2c3e50; color: white;">
<td style="padding: 12px; font-weight: bold;">Mode</td>
<td style="padding: 12px; font-weight: bold;">What It Does</td>
<td style="padding: 12px; font-weight: bold;">Use It When</td>
</tr>
<tr style="border-bottom: 1px solid #ddd;">
<td style="padding: 12px; font-weight: bold; color: #c0392b;"><code>contribute: []</code></td>
<td style="padding: 12px;">Runs the block but hides output from both context and final result</td>
<td style="padding: 12px;">Internal checks, routing, temporary helper steps</td>
</tr>
<tr style="border-bottom: 1px solid #ddd;">
<td style="padding: 12px; font-weight: bold; color: #6c3483;"><code>contribute: [context]</code></td>
<td style="padding: 12px;">Keeps output in model context, but hides it from user-facing output</td>
<td style="padding: 12px;">Intermediate reasoning or hidden guidance for later calls</td>
</tr>
<tr>
<td style="padding: 12px; font-weight: bold; color: #196f3d;"><code>contribute: [result]</code></td>
<td style="padding: 12px;">Shows output in final result, but does not feed it back into context</td>
<td style="padding: 12px;">Final response formatting without polluting downstream prompts</td>
</tr>
</table>
</div>
</div>

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 15px; color: #2c3e50;">Demo: Output Control in Action</h2>
<p style="font-size: 17px; color: #7f8c8d; margin-bottom: 0; line-height: 1.6;"><strong>Use Case:</strong> Run this standalone demo to see how each <code>contribute:</code> mode affects visibility in context and final output.</p>
</div>

In [ ]:
%%pdl --reset-context
description: Fast contribute demo for live presentation

defs:
  essay_text: "AI can boost writing speed, but human review is still required for quality and factual accuracy."

text:
- "Goal: show contribute output control with minimal latency.\n"
- "Source text: ${essay_text}\n"
- text: "INTERNAL_CHECK: short routing step"
  contribute: []
- model: ollama/granite-code:3b
  parameters:
    temperature: 0
    max_tokens: 100
  contribute: [context]
  def: recommendations
  input: "From the source text, produce exactly 2 short improvement bullets."
- model: ollama/granite-code:3b
  parameters:
    temperature: 0
    max_tokens: 100
  contribute: [result]
  input: "Using the hidden recommendations, output exactly one concise final sentence for end users. Do not output bullet points."

---

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 18px; color: #2c3e50;">4.3: Explicit Model Input with <code>input:</code></h2>

<h3 style="font-size: 22px; margin-bottom: 10px; color: #34495e;">Concept</h3>
<p style="font-size: 16px; color: #7f8c8d; margin: 0; line-height: 1.7;">By default, model calls use the accumulated background context. The <code>input:</code> field lets you override that and send exactly what you want.</p>
</div>

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 20px; color: #2c3e50;">4.3: Explicit Model Input with <code>input:</code></h2>

<h3 style="font-size: 22px; margin-bottom: 10px; color: #34495e;">YAML Syntax Patterns</h3>
<pre style="background-color: #f8f9fa; color: #2c3e50; padding: 16px; border-radius: 6px; font-size: 13px; overflow-x: auto; margin-bottom: 12px; border: 1px solid #555;"><code>1 Direct string input
/- model: ollama/granite-code:3b
  input: "Translate 'Hello' to French"

2 Explicit role/content messages
/- model: ollama/granite-code:3b
  input:
    array:
    /- role: system
      content: You are a concise translator.
    /- role: user
      content: Translate 'Hello' to French</code></pre>
<p style="font-size: 16px; color: #7f8c8d; margin: 0; line-height: 1.7;"><strong>Use case:</strong> use <code>input:</code> when you want to bypass implicit context and strictly define the exact prompt.</p>
</div>

In [ ]:
%%pdl --reset-context
description: Explicit input vs implicit context
text:
- "Background context line that should NOT influence explicit input call.\n"
- model: ollama/granite-code:3b
  input: "Translate 'Hello' to French. Return one word only."
  def: direct_input
  contribute: []
- model: ollama/granite-code:3b
  input:
    array:
    - role: system
      content: You are a strict translator. Output a single French word.
    - role: user
      content: Translate 'Thank you' to French.
  def: message_input
  contribute: []
- "Direct input result: ${direct_input}\n"
- "Message array result: ${message_input}\n"

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 20px; color: #2c3e50;">4.3: Explicit Model Input with <code>input:</code></h2>

<h3 style="font-size: 22px; margin-bottom: 10px; color: #34495e;">Use When</h3>
<ul style="font-size: 16px; line-height: 1.9; margin: 0; color: #2c3e50; padding-left: 24px;">
<li>You need strict isolation from previously accumulated context</li>
<li>You want exact role-controlled prompts for reproducibility</li>
<li>You are debugging prompt behavior and need predictable inputs</li>
<li>You are building production flows where prompt payload must be explicit</li>
</ul>
</div>

---

<div style="text-align: center; padding: 50px 20px;">
<h1 style="font-size: 44px; margin-bottom: 10px; color: #2c3e50; font-weight: 400;">SECTION 5: Advanced Features - AutoPDL</h1>
<p style="text-align: center; font-size: 18px; color: #7f8c8d; margin: 0;">Automated Prompt Optimization</p>
<div style="width: 120px; height: 3px; background-color: #3498db; margin: 25px auto;"></div>
</div>

<div style="padding: 40px; max-width: 1000px; margin: 0 auto; margin-top: 30px;">
<h2 style="font-size: 32px; margin-bottom: 30px; color: #2c3e50;">5.1: What is AutoPDL?</h2>
<p style="font-size: 17px; color: #7f8c8d; margin-bottom: 25px; line-height: 1.6;"><strong>AutoPDL = Automated Prompt Optimization:</strong> Instead of manually tuning prompts, AutoPDL systematically searches for the best configuration values. It tests combinations of models, demonstration counts, and strategies to find what works best.</p>

<div style="padding: 25px; background-color: #eafaf1; border-radius: 8px; border-left: 5px solid #27ae60; margin-top: 25px;">
<h3 style="font-size: 20px; margin-top: 0; margin-bottom: 18px; color: #196f3d;">Real Example: Grammar Correction</h3>
<div style="display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 18px;">
<div style="padding: 15px; background-color: rgba(255,255,255,0.5); border-radius: 5px;">
<p style="font-size: 14px; font-weight: bold; margin: 0 0 8px 0; color: #2c3e50;">Models</p>
<p style="font-size: 13px; margin: 0; color: #7f8c8d;">Granite vs GPT-OSS</p>
</div>
<div style="padding: 15px; background-color: rgba(255,255,255,0.5); border-radius: 5px;">
<p style="font-size: 14px; font-weight: bold; margin: 0 0 8px 0; color: #2c3e50;">Examples</p>
<p style="font-size: 13px; margin: 0; color: #7f8c8d;">0, 3, or 5 demos</p>
</div>
<div style="padding: 15px; background-color: rgba(255,255,255,0.5); border-radius: 5px;">
<p style="font-size: 14px; font-weight: bold; margin: 0 0 8px 0; color: #2c3e50;">Strategies</p>
<p style="font-size: 13px; margin: 0; color: #7f8c8d;">Single or verify</p>
</div>
</div>
<p style="font-size: 14px; margin: 20px 0 0 0; color: #196f3d;"><strong>AutoPDL tests 2 × 3 × 2 = 12 combinations</strong> and finds the best.</p>
</div>
</div>

---

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 30px; color: #2c3e50;">5.2: When to Use AutoPDL & Competitive Landscape</h2>

<h3 style="font-size: 22px; margin-bottom: 20px; color: #34495e;">How Does AutoPDL Compare to Other Approaches?</h3>

<div style="display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 18px; margin-bottom: 0;">
<div style="padding: 18px; background-color: #ecf0f1; border-radius: 8px; border-left: 4px solid #7f8c8d;">
<p style="font-size: 13px; font-weight: bold; margin: 0 0 8px 0; color: #2c3e50;">Manual Tuning</p>
<p style="font-size: 13px; margin: 0; color: #7f8c8d;">Hand-adjust prompts, models, examples. Slow, subjective, hard to reproduce.</p>
</div>
<div style="padding: 18px; background-color: #ecf0f1; border-radius: 8px; border-left: 4px solid #7f8c8d;">
<p style="font-size: 13px; font-weight: bold; margin: 0 0 8px 0; color: #2c3e50;">DSPy Optimization</p>
<p style="font-size: 13px; margin: 0; color: #7f8c8d;">Framework for optimizing prompt components. Requires rewriting your program in DSPy's module system.</p>
</div>
<div style="padding: 18px; background-color: #d5f4e6; border-radius: 8px; border-left: 4px solid #27ae60;">
<p style="font-size: 13px; font-weight: bold; margin: 0 0 8px 0; color: #196f3d;">AutoPDL (PDL)</p>
<p style="font-size: 13px; margin: 0; color: #27ae60;">Built-in optimization using successive halving algorithm. 
Can optimize parameters AND prompting patterns (e.g., verify strategy).</p>
</div>
</div>
</div>

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 30px; color: #2c3e50;">5.2: When to Use AutoPDL & Competitive Landscape</h2>

<h3 style="font-size: 22px; margin-bottom: 20px; color: #34495e;">When to Use AutoPDL</h3>

<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 25px;">
<div style="padding: 25px; background-color: #eafaf1; border-radius: 8px; border-left: 5px solid #27ae60;">
<h4 style="font-size: 18px; margin-top: 0; margin-bottom: 15px; color: #196f3d;">Use AutoPDL When:</h4>
<ul style="font-size: 16px; line-height: 2.1; margin: 0; padding-left: 25px; color: #2c3e50;">
<li>Working PDL program exists</li>
<li>Have labeled dataset available</li>
<li>Want to optimize performance</li>
<li>Can define a scoring metric</li>
</ul>
</div>

<div style="padding: 25px; background-color: #fdeaea; border-radius: 8px; border-left: 5px solid #e74c3c;">
<h4 style="font-size: 18px; margin-top: 0; margin-bottom: 15px; color: #c0392b;">Not Appropriate When:</h4>
<ul style="font-size: 16px; line-height: 2.1; margin: 0; padding-left: 25px; color: #2c3e50;">
<li>Still in exploration phase (just prototyping)</li>
<li>No labeled benchmark dataset available</li>
<li>No clear success metric to optimize</li>
<li>Variables/search space not yet defined</li>
</ul>
</div>
</div>
</div>

---

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 30px; color: #2c3e50;">5.3: AutoPDL Workflow</h2>

<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 25px;">
<div style="padding: 20px; background-color: #ecf0f1; border-radius: 8px;">
<h4 style="font-size: 17px; margin-top: 0; margin-bottom: 15px; color: #34495e;">Setup Phase</h4>
<ol style="font-size: 15px; line-height: 2; margin: 0; padding-left: 25px; color: #2c3e50;">
<li>Write PDL with free variables</li>
<li>Prepare dataset (JSONL)</li>
<li>Define scoring function</li>
</ol>
</div>

<div style="padding: 20px; background-color: #ecf0f1; border-radius: 8px;">
<h4 style="font-size: 17px; margin-top: 0; margin-bottom: 15px; color: #34495e;">Execution & Result</h4>
<ol style="font-size: 15px; line-height: 2; margin: 0; padding-left: 25px; color: #2c3e50;">
<li>Configure search space (YAML)</li>
<li>Run: `pdl-optimize -c config.yml`</li>
<li>Get `optimized_program.pdl`</li>
</ol>
</div>
</div>
</div>

---

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<h2 style="font-size: 32px; margin-bottom: 30px; color: #2c3e50;">5.4: AutoPDL Example Walkthrough</h2>

<div style="padding: 20px; background-color: #f8f9fa; border-radius: 8px; margin-bottom: 25px; border-left: 5px solid #3498db;">
<p style="font-size: 18px; font-weight: bold; margin: 0 0 15px 0; color: #2980b9;">Goal: Optimize Grammar Correction</p>
<p style="font-size: 15px; margin: 0; color: #7f8c8d;">Find best model, demonstration count, and strategy for correcting grammar.</p>
</div>

<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 25px;">
<div>
<h4 style="font-size: 18px; margin-bottom: 15px; color: #34495e;">Configuration</h4>
<pre style="color: #f8f8f2; padding: 15px; border-radius: 5px; font-size: 13px; overflow-x: auto; border: 1px solid #555;"><code>variables:
  model:
    - ollama/granite4
    - ollama/gpt-oss
  num_demos: [0, 3, 5]
  verify: [true, false]</code></pre>
</div>

<div style="padding: 20px; background-color: #eafaf1; border-radius: 8px; border-left: 5px solid #27ae60;">
<h4 style="font-size: 18px; margin-top: 0; margin-bottom: 15px; color: #196f3d;">Optimization</h4>
<ul style="font-size: 15px; line-height: 2.1; margin: 0; padding-left: 25px; color: #2c3e50;">
<li><strong>Combinations:</strong> 2 × 3 × 2 = 12</li>
<li><strong>Best:</strong> gpt-oss</li>
<li><strong>Demos:</strong> 3</li>
<li><strong>Verify:</strong> false</li>
</ul>
</div>
</div>
</div>

<div style="text-align: center; padding: 50px 20px;">
<h1 style="font-size: 44px; margin-bottom: 10px; color: #2c3e50; font-weight: 400;">Key Takeaways</h1>
<div style="width: 100px; height: 4px; background-color: #3498db; margin: 20px auto;"></div>
</div>

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<div style="background-color: #ecf0f1; padding: 30px; border-radius: 8px; margin-bottom: 25px;">
<h2 style="font-size: 26px; margin-top: 0; color: #2c3e50; border-bottom: 3px solid #3498db; padding-bottom: 15px; margin-bottom: 20px;">What You Learned</h2>
<ul style="font-size: 18px; line-height: 2; margin: 0; padding-left: 25px;">
<li>PDL makes LLM workflows declarative, reusable, and versionable</li>
<li>Context accumulation and control flow simplify multi-step programs</li>
<li><code>def:</code>, <code>defs:</code>, and <code>contribute:</code> let you control visibility and state cleanly</li>
<li><code>parser:</code> + <code>spec:</code> provide structured, validated outputs for reliability</li>
<li>AutoPDL helps optimize existing programs using benchmark data</li>
</ul>
</div>

<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 25px;">
<div style="padding: 25px; background-color: #d5f4e6; border-radius: 8px; border-left: 5px solid #27ae60;">
<h3 style="font-size: 22px; margin-top: 0; margin-bottom: 15px; color: #196f3d;">When PDL is a Great Fit</h3>
<ul style="font-size: 17px; line-height: 1.9; margin: 0; padding-left: 25px;">
<li>You need reliable structured outputs</li>
<li>You are building multi-step LLM workflows</li>
<li>You want team-friendly, maintainable prompt programs</li>
</ul>
</div>

<div style="padding: 25px; background-color: #fef5e7; border-radius: 8px; border-left: 5px solid #f39c12;">
<h3 style="font-size: 20px; margin-top: 0; margin-bottom: 15px; color: #d68910;">Technical Reflection</h3>
<ol style="font-size: 16px; line-height: 2; margin: 0; padding-left: 25px; color: #2c3e50;">
<li>Which PDL feature gives you the biggest reliability gain?</li>
<li>Where would you add parser/spec first in your current stack?</li>
<li>What part of your workflow should become reusable with defs/functions?</li>
</ol>
</div>
</div>
</div>

---

<div style="text-align: center; padding: 80px 20px;">
<h1 style="font-size: 48px; margin-bottom: 18px; color: #2c3e50; font-weight: 400;">Thank You for Listening</h1>
<p style="font-size: 24px; color: #7f8c8d; margin: 0; text-align:center;">Questions?</p>
<div style="width: 120px; height: 3px; background-color: #3498db; margin: 28px auto 0;"></div>
</div>

<div style="padding: 40px; max-width: 1000px; margin: 0 auto;">
<div style="padding: 25px; background-color: #f4ecf7; border-radius: 8px; border-left: 5px solid #8e44ad; text-align: center; font-size: 17px;">
<p style="margin: 0; color: #6c3483;"><strong>Resources:</strong> <a href="https://github.com/IBM/prompt-declaration-language" style="color: #8e44ad; text-decoration: none; font-weight: 500;">PDL GitHub</a> • <a href="https://github.com/IBM/prompt-declaration-language/wiki" style="color: #8e44ad; text-decoration: none; font-weight: 500;">Docs & Tutorials</a> • <a href="https://github.com/IBM/prompt-declaration-language/discussions" style="color: #8e44ad; text-decoration: none; font-weight: 500;">Community</a></p>
</div>
</div>